In [1]:
import os

DATASET_PATH = r"D:\datasets\dataset"

for root, dirs, files in os.walk(DATASET_PATH):
    print(root)
    break

D:\datasets\dataset


In [2]:
print(os.listdir(DATASET_PATH))

['cataract', 'normal']


In [3]:
import os
import shutil
import random

In [4]:
DATASET_PATH = r"D:\datasets\dataset"

cataract_path = os.path.join(DATASET_PATH, "cataract")
normal_path = os.path.join(DATASET_PATH, "normal")

print(len(os.listdir(cataract_path)), "cataract images")
print(len(os.listdir(normal_path)), "normal images")

1038 cataract images
1074 normal images


In [5]:
split_base = r"D:\datasets\cataract_split"

os.makedirs(os.path.join(split_base, "train", "cataract"), exist_ok=True)
os.makedirs(os.path.join(split_base, "train", "normal"), exist_ok=True)

os.makedirs(os.path.join(split_base, "val", "cataract"), exist_ok=True)
os.makedirs(os.path.join(split_base, "val", "normal"), exist_ok=True)

os.makedirs(os.path.join(split_base, "test", "cataract"), exist_ok=True)
os.makedirs(os.path.join(split_base, "test", "normal"), exist_ok=True)

In [6]:
def split_images(source_folder, class_name):
    
    images = os.listdir(source_folder)
    random.shuffle(images)

    train_split = int(0.7 * len(images))
    val_split = int(0.85 * len(images))

    train_images = images[:train_split]
    val_images = images[train_split:val_split]
    test_images = images[val_split:]

    for img in train_images:
        shutil.copy(os.path.join(source_folder, img),
                    os.path.join(split_base, "train", class_name, img))

    for img in val_images:
        shutil.copy(os.path.join(source_folder, img),
                    os.path.join(split_base, "val", class_name, img))

    for img in test_images:
        shutil.copy(os.path.join(source_folder, img),
                    os.path.join(split_base, "test", class_name, img))

In [7]:
split_images(cataract_path, "cataract")
split_images(normal_path, "normal")

print("Dataset split completed")

Dataset split completed


In [8]:
print(os.listdir(r"D:\datasets\cataract_split"))

['test', 'train', 'val']


In [9]:
DATASET_PATH = r"D:\datasets\cataract_split"

train_dir = os.path.join(DATASET_PATH, "train")
val_dir = os.path.join(DATASET_PATH, "val")
test_dir = os.path.join(DATASET_PATH, "test")

In [10]:
IMG_SIZE = 224
BATCH_SIZE = 16

In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

In [12]:
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [13]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 1477 images belonging to 2 classes.


In [14]:
val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 317 images belonging to 2 classes.


In [15]:
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 318 images belonging to 2 classes.


In [16]:
print(train_data.class_indices)

{'cataract': 0, 'normal': 1}


In [17]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

In [18]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

In [19]:
base_model.trainable = False

In [20]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)
output = layers.Dense(1, activation="sigmoid")(x)

In [21]:
model = models.Model(inputs=base_model.input, outputs=output)

In [22]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [23]:
print("Train images:", train_data.samples)
print("Validation images:", val_data.samples)
print("Test images:", test_data.samples)

Train images: 1477
Validation images: 317
Test images: 318


In [24]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 55s 465ms/step - accuracy: 0.9269 - loss: 0.1843 - val_accuracy: 0.9495 - val_loss: 0.1642
Epoch 2/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 18s 198ms/step - accuracy: 0.9573 - loss: 0.1173 - val_accuracy: 0.9495 - val_loss: 0.1450
Epoch 3/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 18s 198ms/step - accuracy: 0.9614 - loss: 0.1078 - val_accuracy: 0.9590 - val_loss: 0.1175
Epoch 4/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 18s 196ms/step - accuracy: 0.9675 - loss: 0.0928 - val_accuracy: 0.9558 - val_loss: 0.1206
Epoch 5/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 199ms/step - accuracy: 0.9628 - loss: 0.0959 - val_accuracy: 0.9621 - val_loss: 0.1113
Epoch 6/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 201ms/step - accuracy: 0.9580 - loss: 0.0917 - val_accuracy: 0.9653 - val_loss: 0.1142
Epoch 7/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 202ms/step - accuracy: 0.9695 - loss: 0.0788 - val_accuracy: 0.9558 - val_loss: 0.1250
Epoch 8/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 198ms/step - accuracy: 0.9777 - loss: 0.0647 - val_accu

In [25]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test accuracy:", test_accuracy)

20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 390ms/step - accuracy: 0.9717 - loss: 0.0663
Test accuracy: 0.9716981053352356


In [26]:
import numpy as np

y_pred = model.predict(test_data)
pred_classes = (y_pred > 0.5).astype(int)

print("Unique predictions:", np.unique(pred_classes))

20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 373ms/step
Unique predictions: [0 1]


In [27]:
from sklearn.metrics import roc_curve, auc

y_true = test_data.classes

fpr, tpr, thresholds = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

AUC: 0.9988129154795821


In [28]:
from sklearn.metrics import confusion_matrix

y_true = test_data.classes
cm = confusion_matrix(y_true, pred_classes)

print(cm)

[[154   2]
 [  7 155]]


In [29]:
print(train_data.class_indices)

{'cataract': 0, 'normal': 1}


In [30]:
model.save(r"D:\cataract_models\cataract_efficientnetb0.h5")